음식 리뷰 데이터 활용 유사도 검색
- corpus -> embedding vector 유사도 기반 검색

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI


# 1. CSV 파일 불러오기
df = pd.read_csv("./data/fine_food_reviews_1k.csv")  # reviews.csv 파일에는 'review' 컬럼이 있다고 가정
model="text-embedding-3-small"
client = OpenAI()


# 2. 임베딩 벡터 생성 함수
def get_embedding(text, model="text-embedding-3-small"):
    text = text.replace("\n", " ")
    response = client.embeddings.create(
        model=model,
        input=[text]
    )
    return data.embedding for data in response.data


# 3. 모든 리뷰에 대해 임베딩 생성
df["embedding"] = df["Text"].apply(lambda x: get_embedding(x, model))
df.to_csv("./data/review_embedded.csv", index=False, encoding="utf-8")



In [ ]:
import numpy as np

df = pd.read_csv("./data/review_embedded.csv")  # reviews.csv 파일에는 'review' 컬럼이 있다고 가정

# 4. 유사도 검색 함수
def search_reviews(query, top_n=5):
    query_embedding = get_embedding(query)
    query_vector = np.array(query_embedding).reshape(1, -1)

    embeddings = np.array(df['embedding'].tolist())

    # 코사인 유사도 계산
    similarities = cosine_similarity(query_vector, embeddings)[0]
    
    # 상위 N개 리뷰 추출
    df["similarity"] = similarities
    results = df.sort_values("similarity", ascending=False).head(top_n)
    return results[["Text", "similarity"]]


# 5. 사용 예시
query = input("검색어: ")
results = search_reviews(query, top_n=5)
print(results)

                                                  Text  similarity
932  <a href="http://www.amazon.com/gp/product/B006...    0.575234
862  <a href="http://www.amazon.com/gp/product/B006...    0.575234
251  I have a coffee maker that grinds my coffee be...    0.575077
553  I I haven't had a bad cup of coffee yet.  So f...    0.572464
572  Tully's coffee is the smoothest coffee on the ...    0.569305


In [50]:
results.iloc[3]["Text"]

"I I haven't had a bad cup of coffee yet.  So far, my favorites are the Decaf, Columbian, and Breakfast blend.  I only drink one cup of coffee a day, so this type of coffee maker is perfect for me.  Especially since I only like coffee when it's hot and fresh.  My guests love it too!"